In [1]:
%pip install -q pandas numpy scikit-learn scipy tqdm joblib


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# =========================
# CONFIG
# =========================
RAW_STAGE1_PATH = "/kaggle/input/datasets/thuhiuhong/base-clean/stage1_base_clean.csv"
SAVE_DIR = "/kaggle/working"

SEED = 9001
SEQ_LEN = 10

# sequence sampling cho train/val/test gốc
POS_STRIDE = 1
NEG_STRIDE = 6

# thử nhanh hay chạy đầy đủ
QUICK_MODE = False

# ngưỡng quét
THRESHOLD_GRID = np.arange(0.05, 0.96, 0.01)

np.random.seed(SEED)

print("RAW_STAGE1_PATH =", RAW_STAGE1_PATH)
print("SAVE_DIR =", SAVE_DIR)
print("SEQ_LEN =", SEQ_LEN)
print("QUICK_MODE =", QUICK_MODE)


RAW_STAGE1_PATH = /kaggle/input/datasets/thuhiuhong/base-clean/stage1_base_clean.csv
SAVE_DIR = /kaggle/working
SEQ_LEN = 10
QUICK_MODE = False


In [3]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

seed = 9001
np.random.seed(seed)

In [4]:
X = pd.read_csv(RAW_STAGE1_PATH)

In [5]:
X = X.sort_values(['id', 'hour']).reset_index(drop=True)

print(X.shape)
print(X.columns.tolist())
X.head()

(1552210, 41)
['id', 'hour', 'SepsisLabel', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'HospAdmTime', 'ICULOS']


,id,hour,SepsisLabel,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,...,Hct,Hgb,PTT,WBC,Fibrinogen,Platelets,Age,Gender,HospAdmTime,ICULOS
0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,-0.03,1
1,0,1,0,97.0,95.0,NaN,98.0,75.33,NaN,19.0,...,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,-0.03,2
2,0,2,0,89.0,99.0,NaN,122.0,86.00,NaN,22.0,...,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,-0.03,3
3,0,3,0,90.0,95.0,NaN,NaN,NaN,NaN,30.0,...,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,-0.03,4
4,0,4,0,103.0,88.5,NaN,122.0,91.33,NaN,24.5,...,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,-0.03,5


In [6]:
RAW_FEATURES = [
    'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2',
    'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2',
    'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine',
    'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate',
    'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT',
    'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender',
    'HospAdmTime', 'ICULOS'
]

RAW_FEATURES = [col for col in RAW_FEATURES if col in X.columns]

print("Number of raw features:", len(RAW_FEATURES))
print(RAW_FEATURES)

Number of raw features: 38
['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'HospAdmTime', 'ICULOS']


In [7]:
work_df = X[['id', 'hour', 'SepsisLabel'] + RAW_FEATURES].copy()

print(work_df.shape)
work_df.head()

(1552210, 41)


,id,hour,SepsisLabel,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,...,Hct,Hgb,PTT,WBC,Fibrinogen,Platelets,Age,Gender,HospAdmTime,ICULOS
0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,-0.03,1
1,0,1,0,97.0,95.0,NaN,98.0,75.33,NaN,19.0,...,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,-0.03,2
2,0,2,0,89.0,99.0,NaN,122.0,86.00,NaN,22.0,...,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,-0.03,3
3,0,3,0,90.0,95.0,NaN,NaN,NaN,NaN,30.0,...,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,-0.03,4
4,0,4,0,103.0,88.5,NaN,122.0,91.33,NaN,24.5,...,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,-0.03,5


### Get diff

In [8]:
import numpy as np

def get_diff(data, used_cols=None, diff_step=1):
    """
    Calculates feature differences over time while IGNORING NaNs.
    This follows the competition team's logic:
    - remove NaNs in each column
    - compute diff in non-NaN space
    - put values back into original positions
    - positions with original NaN remain NaN
    """
    dims = np.shape(data)
    if len(dims) < 2:
        data = data[:, np.newaxis]
        dims = np.shape(data)

    if used_cols is None:
        used_cols = list(range(dims[1]))

    out = np.zeros((dims[0], len(used_cols)))

    for c in range(len(used_cols)):
        n = used_cols[c]
        col = data[:, n]

        col_noNaNs = col[np.isnan(col) == False]   # remove NaNs
        diff_noNaNs = np.zeros(len(col_noNaNs))

        if len(col_noNaNs) > 0:
            diff_noNaNs[diff_step:] = col_noNaNs[diff_step:] - col_noNaNs[:-diff_step]

        out[np.isnan(col) == False, c] = diff_noNaNs

    out[np.isnan(data[:, used_cols])] = np.nan
    return out

In [9]:
def add_diff_features(data, feature_cols, diff_step=1):
    data = data.copy()

    diff_feature_names = [f'{col}_diff' for col in feature_cols]
    diff_blocks = []

    for pid, g in data.groupby('id', sort=False):
        g = g.sort_values('hour')
        arr = g[feature_cols].values.astype(float)

        diff_arr = get_diff(arr, used_cols=list(range(len(feature_cols))), diff_step=diff_step)
        diff_block = pd.DataFrame(diff_arr, columns=diff_feature_names, index=g.index)
        diff_blocks.append(diff_block)

    diff_df = pd.concat(diff_blocks).sort_index()
    data = pd.concat([data, diff_df], axis=1)

    return data

In [10]:
X_dl = add_diff_features(work_df, RAW_FEATURES, diff_step=1)

print(X_dl.shape)
print("Number of diff features added:", len([c for c in X_dl.columns if c.endswith('_diff')]))
X_dl[['id', 'hour'] + RAW_FEATURES[:3] + [f'{RAW_FEATURES[0]}_diff', f'{RAW_FEATURES[1]}_diff', f'{RAW_FEATURES[2]}_diff']].head(10)

(1552210, 79)
Number of diff features added: 38


,id,hour,HR,O2Sat,Temp,HR_diff,O2Sat_diff,Temp_diff
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,0,1,97.0,95.0,NaN,0.0,0.0,NaN
2,0,2,89.0,99.0,NaN,-8.0,4.0,NaN
3,0,3,90.0,95.0,NaN,1.0,-4.0,NaN
4,0,4,103.0,88.5,NaN,13.0,-6.5,NaN
5,0,5,110.0,91.0,NaN,7.0,2.5,NaN
6,0,6,108.0,92.0,36.11,-2.0,1.0,0.0
7,0,7,106.0,90.5,NaN,-2.0,-1.5,NaN
8,0,8,104.0,95.0,NaN,-2.0,4.5,NaN
9,0,9,102.0,91.0,NaN,-2.0,-4.0,NaN


In [11]:
diff_cols = [c for c in X_dl.columns if c.endswith('_diff')]
print(len(diff_cols))
print(diff_cols)

38
['HR_diff', 'O2Sat_diff', 'Temp_diff', 'SBP_diff', 'MAP_diff', 'DBP_diff', 'Resp_diff', 'EtCO2_diff', 'BaseExcess_diff', 'HCO3_diff', 'FiO2_diff', 'pH_diff', 'PaCO2_diff', 'SaO2_diff', 'AST_diff', 'BUN_diff', 'Alkalinephos_diff', 'Calcium_diff', 'Chloride_diff', 'Creatinine_diff', 'Bilirubin_direct_diff', 'Glucose_diff', 'Lactate_diff', 'Magnesium_diff', 'Phosphate_diff', 'Potassium_diff', 'Bilirubin_total_diff', 'TroponinI_diff', 'Hct_diff', 'Hgb_diff', 'PTT_diff', 'WBC_diff', 'Fibrinogen_diff', 'Platelets_diff', 'Age_diff', 'Gender_diff', 'HospAdmTime_diff', 'ICULOS_diff']


### Missingness feature: `last_reliable`

In [12]:
def last_reliable(data_in):
    dims = np.shape(data_in)
    out = np.zeros(dims)

    non_nans = np.invert(np.isnan(data_in))
    counter = np.zeros(dims[1])

    for row in range(dims[0]):
        counter = counter + 1
        counter[non_nans[row, :]] = 0
        out[row, :] = counter

    return out

In [13]:
def add_last_reliable_features(data, feature_cols):
    data = data.copy()

    rel_feature_names = [f'{col}_last_gap' for col in feature_cols]
    rel_blocks = []

    for pid, g in data.groupby('id', sort=False):
        g = g.sort_values('hour')
        arr = g[feature_cols].values.astype(float)

        rel_arr = last_reliable(arr)
        rel_block = pd.DataFrame(rel_arr, columns=rel_feature_names, index=g.index)
        rel_blocks.append(rel_block)

    rel_df = pd.concat(rel_blocks).sort_index()
    data = pd.concat([data, rel_df], axis=1)

    return data

In [14]:
X_dl = add_last_reliable_features(X_dl, RAW_FEATURES)

print(X_dl.shape)
print("Number of last_gap features added:", len([c for c in X_dl.columns if c.endswith('_last_gap')]))

X_dl[['id', 'hour', 'HR', 'O2Sat', 'Temp', 'HR_last_gap', 'O2Sat_last_gap', 'Temp_last_gap']].head(12)

(1552210, 117)
Number of last_gap features added: 38


,id,hour,HR,O2Sat,Temp,HR_last_gap,O2Sat_last_gap,Temp_last_gap
0,0,0,NaN,NaN,NaN,1.0,1.0,1.0
1,0,1,97.0,95.0,NaN,0.0,0.0,2.0
2,0,2,89.0,99.0,NaN,0.0,0.0,3.0
3,0,3,90.0,95.0,NaN,0.0,0.0,4.0
4,0,4,103.0,88.5,NaN,0.0,0.0,5.0
5,0,5,110.0,91.0,NaN,0.0,0.0,6.0
6,0,6,108.0,92.0,36.11,0.0,0.0,0.0
7,0,7,106.0,90.5,NaN,0.0,0.0,1.0
8,0,8,104.0,95.0,NaN,0.0,0.0,2.0
9,0,9,102.0,91.0,NaN,0.0,0.0,3.0


### Get qSOFA

In [15]:
def get_qSOFA(data_in):
    resp = data_in[:, 6]
    resp_high = (resp >= 22).astype(int)

    sbp = data_in[:, 3]
    sbp_low = (sbp <= 100).astype(int)

# Resp >= 22`: indicates abnormally fast breathing, which can reflect physiological stress, hypoxia, or systemic infection.
# SBP <= 100`: indicates low systolic blood pressure, which may reflect circulatory compromise and poor perfusion.
    return np.stack([resp_high, sbp_low], 1), ["resp_high_qsofa", "sbp_low_qsofa"]

In [16]:
def add_qsofa_features(data, raw_feature_cols):
    data = data.copy()

    arr = data[raw_feature_cols].values.astype(float)
    qsofa_arr, qsofa_labels = get_qSOFA(arr)

    qsofa_df = pd.DataFrame(qsofa_arr, columns=qsofa_labels, index=data.index)
    data = pd.concat([data, qsofa_df], axis=1)

    return data

In [17]:
X_dl = add_qsofa_features(X_dl, RAW_FEATURES)

print(X_dl.shape)
print([c for c in X_dl.columns if 'qsofa' in c.lower()])

X_dl[['id', 'hour', 'Resp', 'SBP', 'resp_high_qsofa', 'sbp_low_qsofa']].head(12)

(1552210, 119)
['resp_high_qsofa', 'sbp_low_qsofa']


,id,hour,Resp,SBP,resp_high_qsofa,sbp_low_qsofa
0,0,0,NaN,NaN,0,0
1,0,1,19.0,98.0,0,1
2,0,2,22.0,122.0,1,0
3,0,3,30.0,NaN,1,0
4,0,4,24.5,122.0,1,0
5,0,5,22.0,NaN,1,0
6,0,6,29.0,123.0,1,0
7,0,7,29.0,93.0,1,1
8,0,8,26.0,133.0,1,0
9,0,9,30.0,134.0,1,0


### SOFA-inspired features

In [18]:
def _over24(col_in):
    le = int(np.ceil(len(col_in) / 24) * 24)
    col = np.zeros(le)
    col[:len(col_in)] = col_in[:, 0]
    col = np.reshape(col, (24, -1))

    col_max = np.max(col, 0)

    col = np.repeat(col_max[np.newaxis, :], 24, 0)
    col = col.flatten()
    col = col[:len(col_in)]

    diff = np.append([0], col_max[1:] - col_max[:-1])
    diff = np.repeat(diff[np.newaxis, :], 24, 0)
    diff = diff.flatten()
    diff = diff[:len(col_in)]

    return col[:, np.newaxis], diff[:, np.newaxis]

In [19]:
from scipy.interpolate import interp1d

def get_SOFA(data_in):
    # Respiratory
    O2Sat = data_in[:, 1].copy()
    O2Sat[O2Sat < 0] = 0

    x = [0, 32, 50, 67, 75, 84, 90, 95, 98, 100]
    y = [2, 20, 28, 35, 40, 50, 60, 70, 80, 100]
    PaO2 = interp1d(x, y, kind='linear', bounds_error=False, fill_value='extrapolate')(O2Sat)

    FiO2 = data_in[:, 10].copy()
    FiO2[FiO2 <= 0] = 0.001

    PaFi = PaO2 / FiO2
    PaFi = np.concatenate([PaFi[:, np.newaxis], 500 * np.ones((len(data_in[:, 0]), 1))], 1)
    PaFi = np.nanmin(PaFi, 1)

    PaFi_points = interp1d(
        [0, 100, 200, 300, 400, 100000],
        [4, 3, 2, 1, 0, 0],
        kind='previous',
        bounds_error=False,
        fill_value=(4, 0)
    )(PaFi)

    PaFi_points[np.isnan(PaFi_points)] = 0
    PaFi_points = PaFi_points[:, np.newaxis]
    PaFi_worst, PaFi_diff = _over24(PaFi_points)

    # Cardiovascular
    MAP = data_in[:, 4].copy()
    MAP[MAP < 0] = 0

    MAP_points = interp1d(
        [0, 70, 1000],
        [1, 0, 0],
        kind='previous',
        bounds_error=False,
        fill_value=(1, 0)
    )(MAP)

    MAP_points[np.isnan(MAP_points)] = 0
    MAP_points = MAP_points[:, np.newaxis]
    MAP_worst, MAP_diff = _over24(MAP_points)

    # Liver
    Liver = data_in[:, 20].copy()
    Liver[Liver < 0] = 0

    Liver_points = interp1d(
        [0, 1.2, 2, 6, 12, 1000],
        [0, 1, 2, 3, 4, 4],
        kind='previous',
        bounds_error=False,
        fill_value=(0, 4)
    )(Liver)

    Liver_points[np.isnan(Liver_points)] = 0
    Liver_points = Liver_points[:, np.newaxis]
    Liver_worst, Liver_diff = _over24(Liver_points)

    # Platelets
    Platelets = data_in[:, 33].copy()
    Platelets[Platelets < 0] = 0

    Platelet_points = interp1d(
        [0, 20, 50, 100, 150, 10000],
        [4, 3, 2, 1, 0, 0],
        kind='previous',
        bounds_error=False,
        fill_value=(4, 0)
    )(Platelets)

    Platelet_points[np.isnan(Platelet_points)] = 0
    Platelet_points = Platelet_points[:, np.newaxis]
    Platelet_worst, Platelet_diff = _over24(Platelet_points)

    # Kidney
    Kidney = data_in[:, 19].copy()
    Kidney[Kidney < 0] = 0

    Kidney_points = interp1d(
        [0, 1.2, 2, 3.5, 5, 1000],
        [0, 1, 2, 3, 4, 4],
        kind='previous',
        bounds_error=False,
        fill_value=(0, 4)
    )(Kidney)

    Kidney_points[np.isnan(Kidney_points)] = 0
    Kidney_points = Kidney_points[:, np.newaxis]
    Kidney_worst, Kidney_diff = _over24(Kidney_points)

    # Total SOFA
    SOFA = PaFi_worst + MAP_worst + Liver_worst + Platelet_worst + Kidney_worst
    SOFA_24, SOFA_diff = _over24(SOFA)

    data_out = [
        PaFi_points, PaFi_worst, PaFi_diff,
        MAP_points, MAP_worst, MAP_diff,
        Liver_points, Liver_worst, Liver_diff,
        Platelet_points, Platelet_worst, Platelet_diff,
        Kidney_points, Kidney_worst, Kidney_diff,
        SOFA_24, SOFA_diff
    ]
    
    labels = [
    "PaFi_points", "PaFi_worst", "PaFi_diff",
    "MAP_sofa_points", "MAP_sofa_worst", "MAP_sofa_diff",
    "Liver_points", "Liver_worst", "Liver_diff",
    "Platelet_points", "Platelet_worst", "Platelet_diff",
    "Kidney_points", "Kidney_worst", "Kidney_diff",
    "SOFA_24", "SOFA_diff"
    ]

    return np.concatenate(data_out, 1), labels

In [20]:
def add_sofa_features(data, raw_feature_cols):
    data = data.copy()

    sofa_blocks = []
    sofa_labels = None

    for pid, g in data.groupby('id', sort=False):
        g = g.sort_values('hour')
        arr = g[raw_feature_cols].values.astype(float)

        sofa_arr, sofa_labels = get_SOFA(arr)
        sofa_block = pd.DataFrame(sofa_arr, columns=sofa_labels, index=g.index)
        sofa_blocks.append(sofa_block)

    sofa_df = pd.concat(sofa_blocks).sort_index()
    data = pd.concat([data, sofa_df], axis=1)

    return data

In [21]:
X_dl = add_sofa_features(X_dl, RAW_FEATURES)

print(X_dl.shape)

sofa_cols = [c for c in X_dl.columns if c in [
    "PaFi_points", "PaFi_worst", "PaFi_diff",
    "MAP_sofa_points", "MAP_sofa_worst", "MAP_sofa_diff",
    "Liver_points", "Liver_worst", "Liver_diff",
    "Platelet_points", "Platelet_worst", "Platelet_diff",
    "Kidney_points", "Kidney_worst", "Kidney_diff",
    "SOFA_24", "SOFA_diff"
]]

print("Number of SOFA features added:", len(sofa_cols))
print(sofa_cols)

X_dl[['id', 'hour'] + sofa_cols[:6]].head(12)

(1552210, 136)
Number of SOFA features added: 17
['PaFi_points', 'PaFi_worst', 'PaFi_diff', 'MAP_sofa_points', 'MAP_sofa_worst', 'MAP_sofa_diff', 'Liver_points', 'Liver_worst', 'Liver_diff', 'Platelet_points', 'Platelet_worst', 'Platelet_diff', 'Kidney_points', 'Kidney_worst', 'Kidney_diff', 'SOFA_24', 'SOFA_diff']


,id,hour,PaFi_points,PaFi_worst,PaFi_diff,MAP_sofa_points,MAP_sofa_worst,MAP_sofa_diff
0,0,0,0.0,2.0,0.0,0.0,1.0,0.0
1,0,1,0.0,2.0,0.0,0.0,1.0,0.0
2,0,2,0.0,2.0,0.0,0.0,1.0,0.0
3,0,3,0.0,2.0,0.0,0.0,1.0,0.0
4,0,4,2.0,2.0,0.0,0.0,1.0,0.0
5,0,5,0.0,2.0,0.0,0.0,1.0,0.0
6,0,6,0.0,2.0,0.0,0.0,1.0,0.0
7,0,7,0.0,2.0,0.0,0.0,1.0,0.0
8,0,8,0.0,2.0,0.0,0.0,1.0,0.0
9,0,9,0.0,2.0,0.0,0.0,1.0,0.0


### Number of features

In [22]:
EXCLUDE_COLS = ['id', 'hour', 'SepsisLabel']
FEATURES = [c for c in X_dl.columns if c not in EXCLUDE_COLS]

print("Number of final DL features:", len(FEATURES))
print(FEATURES[:20])
print(FEATURES[-20:])

Number of final DL features: 133
['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine']
['ICULOS_last_gap', 'resp_high_qsofa', 'sbp_low_qsofa', 'PaFi_points', 'PaFi_worst', 'PaFi_diff', 'MAP_sofa_points', 'MAP_sofa_worst', 'MAP_sofa_diff', 'Liver_points', 'Liver_worst', 'Liver_diff', 'Platelet_points', 'Platelet_worst', 'Platelet_diff', 'Kidney_points', 'Kidney_worst', 'Kidney_diff', 'SOFA_24', 'SOFA_diff']


### Check missing column

In [23]:
missing_counts = X_dl[FEATURES].isnull().sum().sort_values(ascending=False)

print("Total missing values:", int(X_dl[FEATURES].isnull().sum().sum()))
print(missing_counts.head(20))

Total missing values: 84576454
Bilirubin_direct         1549220
Bilirubin_direct_diff    1549220
Fibrinogen               1541968
Fibrinogen_diff          1541968
TroponinI_diff           1537429
TroponinI                1537429
Bilirubin_total          1529069
Bilirubin_total_diff     1529069
Alkalinephos_diff        1527269
Alkalinephos             1527269
AST_diff                 1527027
AST                      1527027
Lactate                  1510764
Lactate_diff             1510764
PTT                      1506511
PTT_diff                 1506511
SaO2_diff                1498649
SaO2                     1498649
EtCO2_diff               1494574
EtCO2                    1494574
dtype: int64


### fill NaN = 0

In [24]:
X_dl[FEATURES] = X_dl[FEATURES].fillna(0)

print("Remaining missing values after final fill:", int(X_dl[FEATURES].isnull().sum().sum()))

Remaining missing values after final fill: 0


### Split train/val/test set

In [25]:
from sklearn.model_selection import GroupShuffleSplit

In [26]:
patient_df = X_dl[['id']].drop_duplicates().copy()

print("Number of patients:", patient_df.shape[0])
patient_df.head()

Number of patients: 40336


,id
0,0
54,1
77,2
125,3
154,4


In [27]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(gss1.split(patient_df, groups=patient_df['id']))

train_ids = patient_df.iloc[train_idx]['id'].values
test_ids  = patient_df.iloc[test_idx]['id'].values

print("Train patient count:", len(train_ids))
print("Test patient count:", len(test_ids))

Train patient count: 32268
Test patient count: 8068


In [28]:
train_val_df = X_dl[X_dl['id'].isin(train_ids)].copy()
test_df      = X_dl[X_dl['id'].isin(test_ids)].copy()

print("train_val_df shape:", train_val_df.shape)
print("test_df shape:", test_df.shape)

train_val_df shape: (1241213, 136)
test_df shape: (310997, 136)


In [29]:
tv_patients = train_val_df[['id']].drop_duplicates().copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx2, val_idx = next(gss2.split(tv_patients, groups=tv_patients['id']))

train_ids_final = tv_patients.iloc[train_idx2]['id'].values
val_ids         = tv_patients.iloc[val_idx]['id'].values

print("Final train patient count:", len(train_ids_final))
print("Val patient count:", len(val_ids))

Final train patient count: 25814
Val patient count: 6454


In [30]:
train_df = train_val_df[train_val_df['id'].isin(train_ids_final)].copy()
val_df   = train_val_df[train_val_df['id'].isin(val_ids)].copy()

print("train_df shape:", train_df.shape)
print("val_df shape:", val_df.shape)
print("test_df shape:", test_df.shape)

train_df shape: (992387, 136)
val_df shape: (248826, 136)
test_df shape: (310997, 136)


In [31]:
train_id_set = set(train_df['id'].unique())
val_id_set   = set(val_df['id'].unique())
test_id_set  = set(test_df['id'].unique())

print("Train ∩ Val:", len(train_id_set & val_id_set))
print("Train ∩ Test:", len(train_id_set & test_id_set))
print("Val ∩ Test:", len(val_id_set & test_id_set))

Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


### fit scaler

In [32]:
from sklearn.preprocessing import PowerTransformer

In [33]:
scaler = PowerTransformer()

train_df[FEATURES] = scaler.fit_transform(train_df[FEATURES])
val_df[FEATURES]   = scaler.transform(val_df[FEATURES])
test_df[FEATURES]  = scaler.transform(test_df[FEATURES])

print(train_df.shape, val_df.shape, test_df.shape)

(992387, 136) (248826, 136) (310997, 136)


In [34]:
print(train_df[FEATURES[:10]].describe().T[['mean', 'std']])

                    mean       std
HR          8.605384e-16  1.000001
O2Sat       4.603839e-17  1.000001
Temp       -8.571875e-17  1.000001
SBP         1.492875e-15  1.000001
MAP        -7.474973e-16  1.000001
DBP         1.261294e-16  1.000001
Resp       -5.148280e-16  1.000001
EtCO2      -3.173284e-17  1.000001
BaseExcess  2.676742e-17  1.000001
HCO3       -2.067790e-17  1.000001


### Save scale

In [35]:
train_df.to_csv('/kaggle/working/train_df_stage2_dl.csv', index=False)
val_df.to_csv('/kaggle/working/val_df_stage2_dl.csv', index=False)
test_df.to_csv('/kaggle/working/test_df_stage2_dl.csv', index=False)

print("Saved scaled train/val/test dataframes.")

Saved scaled train/val/test dataframes.


###  Build sequences for the DL

In [36]:
import numpy as np
from tqdm import tqdm

def create_sequences(data, feature_cols, seq_len=SEQ_LEN, pos_stride=POS_STRIDE, neg_stride=NEG_STRIDE):
    X_seq, y_seq, id_seq = [], [], []

    for pid, g in tqdm(data.groupby('id'), total=data['id'].nunique()):
        g = g.sort_values('hour')

        feat = g[feature_cols].values
        label = g['SepsisLabel'].values

        n = len(g)
        i = 0

        while i <= n - seq_len:
            x_window = feat[i:i+seq_len]
            y_window = label[i+seq_len-1]   # lấy nhãn ở cuối cửa sổ

            X_seq.append(x_window)
            y_seq.append(y_window)
            id_seq.append(pid)

            stride = pos_stride if y_window == 1 else neg_stride
            i += stride

    return np.array(X_seq), np.array(y_seq), np.array(id_seq)

In [37]:
X_train, y_train, id_train = create_sequences(train_df, FEATURES, seq_len=SEQ_LEN, pos_stride=1, neg_stride=6)
X_val, y_val, id_val       = create_sequences(val_df, FEATURES, seq_len=SEQ_LEN, pos_stride=1, neg_stride=6)
X_test, y_test, id_test    = create_sequences(test_df, FEATURES, seq_len=SEQ_LEN, pos_stride=1, neg_stride=6)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

100%|██████████| 8068/8068 [00:17<00:00, 469.70it/s]


X_train: (145772, 10, 133) y_train: (145772,)
X_val: (36597, 10, 133) y_val: (36597,)
X_test: (45552, 10, 133) y_test: (45552,)


In [38]:
print("Train positive rate:", y_train.mean())
print("Val positive rate:", y_val.mean())
print("Test positive rate:", y_test.mean())

Train positive rate: 0.0752956672063222
Val positive rate: 0.07631773096155423
Test positive rate: 0.07057867931155602


In [39]:
save_dir = SAVE_DIR

np.save(os.path.join(save_dir, 'X_train.npy'), X_train)
np.save(os.path.join(save_dir, 'y_train.npy'), y_train)
np.save(os.path.join(save_dir, 'id_train.npy'), id_train)

np.save(os.path.join(save_dir, 'X_val.npy'), X_val)
np.save(os.path.join(save_dir, 'y_val.npy'), y_val)
np.save(os.path.join(save_dir, 'id_val.npy'), id_val)

np.save(os.path.join(save_dir, 'X_test.npy'), X_test)
np.save(os.path.join(save_dir, 'y_test.npy'), y_test)
np.save(os.path.join(save_dir, 'id_test.npy'), id_test)

print("Saved standard sequence arrays to:", save_dir)


Saved standard sequence arrays to: /kaggle/working


## Utility-style eval sequences

In [40]:
from tqdm import tqdm

def create_sequences_for_eval(data, feature_cols, seq_len=SEQ_LEN):
    X_seq, y_seq, id_seq, hour_seq = [], [], [], []

    for pid, g in tqdm(data.groupby('id', sort=False), total=data['id'].nunique()):
        g = g.sort_values('hour')

        feat = g[feature_cols].values
        label = g['SepsisLabel'].values
        hours = g['hour'].values

        n = len(g)
        if n < seq_len:
            continue

        for i in range(0, n - seq_len + 1):
            X_seq.append(feat[i:i+seq_len])
            y_seq.append(label[i+seq_len-1])
            id_seq.append(pid)
            hour_seq.append(hours[i+seq_len-1])

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(y_seq, dtype=np.int32),
        np.asarray(id_seq),
        np.asarray(hour_seq)
    )


In [41]:
X_val_eval, y_val_eval, id_val_eval, hour_val_eval = create_sequences_for_eval(
    val_df, FEATURES, seq_len=SEQ_LEN
)

X_test_eval, y_test_eval, id_test_eval, hour_test_eval = create_sequences_for_eval(
    test_df, FEATURES, seq_len=SEQ_LEN
)

print("X_val_eval :", X_val_eval.shape)
print("y_val_eval :", y_val_eval.shape)
print("X_test_eval:", X_test_eval.shape)
print("y_test_eval:", y_test_eval.shape)


100%|██████████| 8068/8068 [00:17<00:00, 467.40it/s]


X_val_eval : (190797, 10, 133)
y_val_eval : (190797,)
X_test_eval: (238447, 10, 133)
y_test_eval: (238447,)


## Model training

In [42]:
import pandas as pd
import numpy as np
import os
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.metrics import Recall

import warnings
warnings.filterwarnings('ignore')

seed = 9001
np.random.seed(seed)

2026-04-09 08:41:14.569424: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775724074.757589      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775724074.815633      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775724075.313129      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775724075.313154      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775724075.313157      24 computation_placer.cc:177] computation placer alr

In [43]:
print("Using in-memory arrays created above:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


Using in-memory arrays created above:
X_train: (145772, 10, 133)
y_train: (145772,)
X_val  : (36597, 10, 133)
y_val  : (36597,)
X_test : (45552, 10, 133)
y_test : (45552,)


In [44]:
import numpy as np

X_train = np.asarray(X_train).astype(np.float32)
X_val   = np.asarray(X_val).astype(np.float32)
X_test  = np.asarray(X_test).astype(np.float32)

y_train = np.asarray(y_train).astype(np.int32).reshape(-1)
y_val   = np.asarray(y_val).astype(np.int32).reshape(-1)
y_test  = np.asarray(y_test).astype(np.int32).reshape(-1)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("y_val:  ", y_val.shape, y_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("y_test: ", y_test.shape, y_test.dtype)

assert X_train.ndim == 3, f"X_train phải là 3D, hiện tại là {X_train.ndim}D"
assert X_val.ndim == 3, f"X_val phải là 3D, hiện tại là {X_val.ndim}D"
assert X_test.ndim == 3, f"X_test phải là 3D, hiện tại là {X_test.ndim}D"

assert len(X_train) == len(y_train), "X_train và y_train lệch số mẫu"
assert len(X_val) == len(y_val), "X_val và y_val lệch số mẫu"
assert len(X_test) == len(y_test), "X_test và y_test lệch số mẫu"

print("timesteps =", X_train.shape[1])
print("n_features =", X_train.shape[2])
print("Train class distribution:", np.bincount(y_train))
print("Val class distribution:", np.bincount(y_val))
print("Test class distribution:", np.bincount(y_test))

X_train: (145772, 10, 133) float32
y_train: (145772,) int32
X_val:   (36597, 10, 133) float32
y_val:   (36597,) int32
X_test:  (45552, 10, 133) float32
y_test:  (45552,) int32
timesteps = 10
n_features = 133
Train class distribution: [134796  10976]
Val class distribution: [33804  2793]
Test class distribution: [42337  3215]


In [45]:
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional

def create_model(n_lstm_units, dropout_rate):
    model = Sequential([
        Bidirectional(
            LSTM(n_lstm_units, return_sequences=True),
            input_shape=(X_train.shape[1], X_train.shape[2])
        ),
        Dropout(dropout_rate),
        BatchNormalization(),

        Bidirectional(
            LSTM(max(n_lstm_units // 2, 8), return_sequences=False)
        ),
        Dropout(dropout_rate),
        BatchNormalization(),

        Dense(16, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(curve='ROC', name='auroc'),
            tf.keras.metrics.AUC(curve='PR', name='auprc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision')
        ]
    )
    return model

print("✅ Đã khởi tạo create_model")

✅ Đã khởi tạo create_model


In [46]:
from sklearn.utils.class_weight import compute_class_weight
import tensorflow.keras.backend as K
import tensorflow as tf

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = {
    int(cls): float(w) for cls, w in zip(np.unique(y_train), class_weights_array)
}
print("Trọng số lớp Train:", class_weights_dict)

if QUICK_MODE:
    lstm_units_options = [32]
    dropout_rate_options = [0.4]
    batch_size_options = [256]
    num_epochs = 3
    final_epochs = 8
else:
    lstm_units_options = [16, 32, 50]
    dropout_rate_options = [0.3, 0.4, 0.5]
    batch_size_options = [256, 512]
    num_epochs = 15
    final_epochs = 50

print("lstm_units_options =", lstm_units_options)
print("dropout_rate_options =", dropout_rate_options)
print("batch_size_options =", batch_size_options)
print("search epochs =", num_epochs)
print("final epochs =", final_epochs)

results = []

for n_units in lstm_units_options:
    for dropout_rate in dropout_rate_options:
        for batch_size in batch_size_options:
            print(f"\nĐang test: units={n_units}, dropout={dropout_rate}, batch_size={batch_size}")

            K.clear_session()

            model = create_model(n_lstm_units=n_units, dropout_rate=dropout_rate)

            history = model.fit(
                X_train, y_train,
                epochs=num_epochs,
                batch_size=batch_size,
                class_weight=class_weights_dict,
                validation_data=(X_val, y_val),
                shuffle=True,
                verbose=0
            )

            val_loss, val_accuracy, val_auroc, val_auprc, val_recall, val_precision = model.evaluate(
                X_val, y_val, verbose=0
            )

            print(
                f"Val AUROC: {val_auroc:.4f} | "
                f"Val AUPRC: {val_auprc:.4f} | "
                f"Recall: {val_recall:.4f} | "
                f"Precision: {val_precision:.4f}"
            )

            results.append((
                n_units, dropout_rate, batch_size,
                val_loss, val_accuracy, val_auroc, val_auprc, val_recall, val_precision
            ))

best_result = sorted(results, key=lambda x: x[5], reverse=True)[0]

print("\n" + "="*60)
print("Best Hyperparameters:")
print(f"- LSTM Units:   {best_result[0]}")
print(f"- Dropout Rate: {best_result[1]}")
print(f"- Batch Size:   {best_result[2]}")
print(f"- Val AUROC:    {best_result[5]:.4f}")
print(f"- Val AUPRC:    {best_result[6]:.4f}")
print("="*60)


Trọng số lớp Train: {0: 0.5407133742841034, 1: 6.64048833819242}
lstm_units_options = [16, 32, 50]
dropout_rate_options = [0.3, 0.4, 0.5]
batch_size_options = [256, 512]
search epochs = 15
final epochs = 50

Đang test: units=16, dropout=0.3, batch_size=256


I0000 00:00:1775724099.546762      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775724099.552806      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1775724107.973597      77 cuda_dnn.cc:529] Loaded cuDNN version 91002


Val AUROC: 0.7826 | Val AUPRC: 0.3357 | Recall: 0.5546 | Precision: 0.3034

Đang test: units=16, dropout=0.3, batch_size=512
Val AUROC: 0.7932 | Val AUPRC: 0.3225 | Recall: 0.5822 | Precision: 0.2795

Đang test: units=16, dropout=0.4, batch_size=256
Val AUROC: 0.7802 | Val AUPRC: 0.3213 | Recall: 0.5521 | Precision: 0.2842

Đang test: units=16, dropout=0.4, batch_size=512
Val AUROC: 0.7985 | Val AUPRC: 0.3175 | Recall: 0.5997 | Precision: 0.2852

Đang test: units=16, dropout=0.5, batch_size=256
Val AUROC: 0.7935 | Val AUPRC: 0.3260 | Recall: 0.5517 | Precision: 0.3022

Đang test: units=16, dropout=0.5, batch_size=512
Val AUROC: 0.8040 | Val AUPRC: 0.3303 | Recall: 0.6452 | Precision: 0.2551

Đang test: units=32, dropout=0.3, batch_size=256
Val AUROC: 0.7472 | Val AUPRC: 0.3035 | Recall: 0.4508 | Precision: 0.3265

Đang test: units=32, dropout=0.3, batch_size=512
Val AUROC: 0.7745 | Val AUPRC: 0.3263 | Recall: 0.5256 | Precision: 0.3046

Đang test: units=32, dropout=0.4, batch_size=256


In [47]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tensorflow.keras.backend as K

K.clear_session()

final_model = create_model(
    n_lstm_units=best_result[0],
    dropout_rate=best_result[1]
)

early_stopping = EarlyStopping(
    monitor='val_auroc',
    mode='max',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_auroc',
    mode='max',
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_lstm_model.keras',
    monitor='val_auroc',
    mode='max',
    save_best_only=True,
    verbose=1
)

print("🚀 ĐANG HUẤN LUYỆN FINAL MODEL...")
history = final_model.fit(
    X_train, y_train,
    epochs=final_epochs,
    batch_size=best_result[2],
    class_weight=class_weights_dict,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr, checkpoint],
    shuffle=True,
    verbose=1
)

🚀 ĐANG HUẤN LUYỆN FINAL MODEL...
Epoch 1/50
282/285 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6360 - auprc: 0.1608 - auroc: 0.6789 - loss: 0.6501 - precision: 0.1242 - recall: 0.6259
Epoch 1: val_auroc improved from -inf to 0.83046, saving model to best_lstm_model.keras
285/285 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.6364 - auprc: 0.1615 - auroc: 0.6800 - loss: 0.6493 - precision: 0.1245 - recall: 0.6272 - val_accuracy: 0.7704 - val_auprc: 0.2876 - val_auroc: 0.8305 - val_loss: 0.4737 - val_precision: 0.2114 - val_recall: 0.7358 - learning_rate: 0.0010
Epoch 2/50
285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7442 - auprc: 0.2890 - auroc: 0.8318 - loss: 0.5086 - precision: 0.1978 - recall: 0.7916
Epoch 2: val_auroc improved from 0.83046 to 0.84178, saving model to best_lstm_model.keras
285/285 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7443 - auprc: 0.2891 - auroc: 0.8319 - loss: 0.5086 - precision: 0.1979 - recall: 0.7917 - val_accuracy: 0.7824 - val_auprc

In [48]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

print("⏳ Đang chấm điểm trên tập Test...")
test_metrics = final_model.evaluate(X_test, y_test, verbose=0)

print("KẾT QUẢ TRÊN TẬP TEST:")
print(f"Test Loss:      {test_metrics[0]:.4f}")
print(f"Test Accuracy:  {test_metrics[1]:.4f}")
print(f"Test AUROC:     {test_metrics[2]:.4f}")
print(f"Test AUPRC:     {test_metrics[3]:.4f}")
print(f"Test Recall:    {test_metrics[4]:.4f}")
print(f"Test Precision: {test_metrics[5]:.4f}")

print("📦 Đang dự đoán trên X_test...")
pred_prob = final_model.predict(X_test, verbose=0).flatten()
pred_label = (pred_prob >= 0.5).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, pred_label))

print("\nClassification Report:")
print(classification_report(y_test, pred_label, digits=4))

print(f"ROC-AUC (sklearn): {roc_auc_score(y_test, pred_prob):.4f}")
print(f"PR-AUC  (sklearn): {average_precision_score(y_test, pred_prob):.4f}")

df_lstm = pd.DataFrame({
    'y_true': y_test,
    'pred_lstm': pred_prob,
    'pred_label': pred_label
})
df_lstm.to_csv('predictions_lstm.csv', index=False)

print("✅ Đã xuất 'predictions_lstm.csv'")

⏳ Đang chấm điểm trên tập Test...
KẾT QUẢ TRÊN TẬP TEST:
Test Loss:      0.3752
Test Accuracy:  0.8232
Test AUROC:     0.8640
Test AUPRC:     0.3481
Test Recall:    0.7530
Test Precision: 0.2501
📦 Đang dự đoán trên X_test...
Confusion Matrix:
[[35078  7259]
 [  794  2421]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9779    0.8285    0.8970     42337
           1     0.2501    0.7530    0.3755      3215

    accuracy                         0.8232     45552
   macro avg     0.6140    0.7908    0.6363     45552
weighted avg     0.9265    0.8232    0.8602     45552

ROC-AUC (sklearn): 0.8640
PR-AUC  (sklearn): 0.3490
✅ Đã xuất 'predictions_lstm.csv'


In [49]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, recall_score

val_prob = final_model.predict(X_val, verbose=0).ravel()

rows = []
for th in np.arange(0.05, 0.96, 0.01):
    val_pred = (val_prob >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, val_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    youden_j = sensitivity + specificity - 1

    rows.append({
        "threshold": round(th, 2),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "youden_j": youden_j
    })

df_th = pd.DataFrame(rows)
best = df_th.loc[df_th["youden_j"].idxmax()]
print(best)

threshold      0.250000
sensitivity    0.803437
specificity    0.730239
youden_j       0.533676
Name: 20, dtype: float64


In [50]:
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

# Tạo xác suất dự đoán trên test
test_prob = final_model.predict(X_test, verbose=0).ravel()

def eval_threshold(y_true, y_prob, th):
    y_pred = (y_prob >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    return {
        "threshold": th,
        "accuracy": acc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

rows = []
for th in [0.30, 0.35, 0.40, 0.45, 0.50]:
    rows.append(eval_threshold(y_test, test_prob, th))

df_compare = pd.DataFrame(rows)
print(df_compare)

   threshold  accuracy  sensitivity  specificity  precision    recall  \
0       0.30  0.764774     0.817418     0.760777   0.206021  0.817418   
1       0.35  0.781854     0.802488     0.780287   0.217135  0.802488   
2       0.40  0.796891     0.788180     0.797553   0.228186  0.788180   
3       0.45  0.810524     0.770140     0.813591   0.238812  0.770140   
4       0.50  0.823213     0.753033     0.828542   0.250103  0.753033   

         f1  
0  0.329096  
1  0.341790  
2  0.353911  
3  0.364573  
4  0.375494  


In [51]:
candidate = df_th[df_th["sensitivity"] >= 0.80].copy()
best = candidate.sort_values("specificity", ascending=False).iloc[0]
print(best)

threshold      0.250000
sensitivity    0.803437
specificity    0.730239
youden_j       0.533676
Name: 20, dtype: float64


In [52]:
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score, accuracy_score

best_threshold = 0.36

test_prob = final_model.predict(X_test, verbose=0).ravel()
test_pred = (test_prob >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)
acc = accuracy_score(y_test, test_pred)
youden_j = sensitivity + specificity - 1

print("=== TEST RESULT WITH THRESHOLD = 0.36 ===")
print(f"Accuracy:    {acc:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-score:    {f1:.4f}")
print(f"Youden's J:  {youden_j:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred, digits=4))

=== TEST RESULT WITH THRESHOLD = 0.36 ===
Accuracy:    0.7856
Sensitivity: 0.8000
Specificity: 0.7845
Precision:   0.2199
Recall:      0.8000
F1-score:    0.3450
Youden's J:  0.5845

Confusion Matrix:
[[33214  9123]
 [  643  2572]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9810    0.7845    0.8718     42337
           1     0.2199    0.8000    0.3450      3215

    accuracy                         0.7856     45552
   macro avg     0.6005    0.7923    0.6084     45552
weighted avg     0.9273    0.7856    0.8346     45552



## PhysioNet utility-style evaluation in memory

Cell dưới đây dùng đúng logic utility của challenge 2019, nhưng chạy **trực tiếp trên arrays** thay vì phải ghi thư mục labels/predictions trước.  
Nó phù hợp hơn với bài khóa luận hiện tại vì bạn đang làm test nội bộ.


In [53]:

import warnings
import numpy as np
import pandas as pd

def compute_auc(labels, predictions, check_errors=True):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)

    if check_errors:
        if len(predictions) != len(labels):
            raise Exception('Numbers of predictions and labels must be the same.')
        for label in labels:
            if label not in (0, 1):
                raise Exception('Labels must satisfy label == 0 or label == 1.')
        for prediction in predictions:
            if not 0 <= prediction <= 1:
                warnings.warn('Predictions do not satisfy 0 <= prediction <= 1.')

    thresholds = np.unique(predictions)[::-1]
    if len(thresholds) == 0:
        return np.nan, np.nan
    if thresholds[0] != 1:
        thresholds = np.insert(thresholds, 0, 1)
    if thresholds[-1] == 0:
        thresholds = thresholds[:-1]

    n = len(labels)
    m = len(thresholds)

    tp = np.zeros(m)
    fp = np.zeros(m)
    fn = np.zeros(m)
    tn = np.zeros(m)

    idx = np.argsort(predictions)[::-1]
    i = 0
    for j in range(m):
        if j == 0:
            tp[j] = 0
            fp[j] = 0
            fn[j] = np.sum(labels)
            tn[j] = n - fn[j]
        else:
            tp[j] = tp[j - 1]
            fp[j] = fp[j - 1]
            fn[j] = fn[j - 1]
            tn[j] = tn[j - 1]

        while i < n and predictions[idx[i]] >= thresholds[j]:
            if labels[idx[i]]:
                tp[j] += 1
                fn[j] -= 1
            else:
                fp[j] += 1
                tn[j] -= 1
            i += 1

    tpr = np.zeros(m)
    tnr = np.zeros(m)
    ppv = np.zeros(m)

    for j in range(m):
        tpr[j] = tp[j] / (tp[j] + fn[j]) if (tp[j] + fn[j]) else 1
        tnr[j] = tn[j] / (fp[j] + tn[j]) if (fp[j] + tn[j]) else 1
        ppv[j] = tp[j] / (tp[j] + fp[j]) if (tp[j] + fp[j]) else 1

    auroc = 0
    auprc = 0
    for j in range(m - 1):
        auroc += 0.5 * (tpr[j + 1] - tpr[j]) * (tnr[j + 1] + tnr[j])
        auprc += (tpr[j + 1] - tpr[j]) * ppv[j + 1]

    return auroc, auprc


def compute_accuracy_f_measure(labels, predictions, check_errors=True):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)

    if check_errors:
        if len(predictions) != len(labels):
            raise Exception('Numbers of predictions and labels must be the same.')
        for label in labels:
            if label not in (0, 1):
                raise Exception('Labels must satisfy label == 0 or label == 1.')
        for prediction in predictions:
            if prediction not in (0, 1):
                raise Exception('Predictions must satisfy prediction == 0 or prediction == 1.')

    tp = fp = fn = tn = 0
    for i in range(len(labels)):
        if labels[i] and predictions[i]:
            tp += 1
        elif (not labels[i]) and predictions[i]:
            fp += 1
        elif labels[i] and (not predictions[i]):
            fn += 1
        else:
            tn += 1

    accuracy = float(tp + tn) / float(tp + fp + fn + tn) if (tp + fp + fn + tn) else 1.0
    f_measure = float(2 * tp) / float(2 * tp + fp + fn) if (2 * tp + fp + fn) else 1.0
    return accuracy, f_measure


def compute_prediction_utility(
    labels, predictions,
    dt_early=-12, dt_optimal=-6, dt_late=3.0,
    max_u_tp=1, min_u_fn=-2, u_fp=-0.05, u_tn=0,
    check_errors=True
):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)

    if check_errors:
        if len(predictions) != len(labels):
            raise Exception('Numbers of predictions and labels must be the same.')
        for label in labels:
            if label not in (0, 1):
                raise Exception('Labels must satisfy label == 0 or label == 1.')
        for prediction in predictions:
            if prediction not in (0, 1):
                raise Exception('Predictions must satisfy prediction == 0 or prediction == 1.')
        if dt_early >= dt_optimal:
            raise Exception('The earliest beneficial time for predictions must be before the optimal time.')
        if dt_optimal >= dt_late:
            raise Exception('The optimal time for predictions must be before the latest beneficial time.')

    if np.any(labels):
        is_septic = True
        t_sepsis = np.argmax(labels) - dt_optimal
    else:
        is_septic = False
        t_sepsis = float('inf')

    n = len(labels)

    m_1 = float(max_u_tp) / float(dt_optimal - dt_early)
    b_1 = -m_1 * dt_early
    m_2 = float(-max_u_tp) / float(dt_late - dt_optimal)
    b_2 = -m_2 * dt_late
    m_3 = float(min_u_fn) / float(dt_late - dt_optimal)
    b_3 = -m_3 * dt_optimal

    u = np.zeros(n)
    for t in range(n):
        if t <= t_sepsis + dt_late:
            if is_septic and predictions[t]:
                if t <= t_sepsis + dt_optimal:
                    u[t] = max(m_1 * (t - t_sepsis) + b_1, u_fp)
                elif t <= t_sepsis + dt_late:
                    u[t] = m_2 * (t - t_sepsis) + b_2
            elif (not is_septic) and predictions[t]:
                u[t] = u_fp
            elif is_septic and (not predictions[t]):
                if t <= t_sepsis + dt_optimal:
                    u[t] = 0
                elif t <= t_sepsis + dt_late:
                    u[t] = m_3 * (t - t_sepsis) + b_3
            elif (not is_septic) and (not predictions[t]):
                u[t] = u_tn

    return np.sum(u)


def evaluate_sepsis_score_from_arrays(ids, labels, probabilities, predictions):
    df = pd.DataFrame({
        "id": ids,
        "SepsisLabel": labels,
        "PredictedProbability": probabilities,
        "PredictedLabel": predictions,
    })

    cohort_labels = []
    cohort_predictions = []
    cohort_probabilities = []

    for _, g in df.groupby("id", sort=False):
        y = g["SepsisLabel"].to_numpy().astype(int)
        p_prob = g["PredictedProbability"].to_numpy().astype(float)
        p_bin = g["PredictedLabel"].to_numpy().astype(int)

        cohort_labels.append(y)
        cohort_predictions.append(p_bin)
        cohort_probabilities.append(p_prob)

    labels_all = np.concatenate(cohort_labels)
    preds_all = np.concatenate(cohort_predictions)
    probs_all = np.concatenate(cohort_probabilities)

    auroc, auprc = compute_auc(labels_all, probs_all)
    accuracy, f_measure = compute_accuracy_f_measure(labels_all, preds_all)

    observed_utilities = np.zeros(len(cohort_labels))
    best_utilities = np.zeros(len(cohort_labels))
    inaction_utilities = np.zeros(len(cohort_labels))

    dt_early = -12
    dt_optimal = -6
    dt_late = 3
    max_u_tp = 1
    min_u_fn = -2
    u_fp = -0.05
    u_tn = 0

    for k in range(len(cohort_labels)):
        y = cohort_labels[k]
        pred = cohort_predictions[k]
        num_rows = len(y)

        best_predictions = np.zeros(num_rows)
        inaction_predictions = np.zeros(num_rows)

        if np.any(y):
            t_sepsis = np.argmax(y) - dt_optimal
            start = max(0, t_sepsis + dt_early)
            end = min(t_sepsis + dt_late + 1, num_rows)
            best_predictions[start:end] = 1

        observed_utilities[k] = compute_prediction_utility(
            y, pred, dt_early, dt_optimal, dt_late, max_u_tp, min_u_fn, u_fp, u_tn
        )
        best_utilities[k] = compute_prediction_utility(
            y, best_predictions, dt_early, dt_optimal, dt_late, max_u_tp, min_u_fn, u_fp, u_tn
        )
        inaction_utilities[k] = compute_prediction_utility(
            y, inaction_predictions, dt_early, dt_optimal, dt_late, max_u_tp, min_u_fn, u_fp, u_tn
        )

    unnormalized_observed_utility = np.sum(observed_utilities)
    unnormalized_best_utility = np.sum(best_utilities)
    unnormalized_inaction_utility = np.sum(inaction_utilities)

    normalized_observed_utility = (
        (unnormalized_observed_utility - unnormalized_inaction_utility)
        / (unnormalized_best_utility - unnormalized_inaction_utility)
    )

    return {
        "auroc": auroc,
        "auprc": auprc,
        "accuracy": accuracy,
        "f_measure": f_measure,
        "utility": normalized_observed_utility
    }


def sweep_thresholds_for_utility(ids, labels, probabilities, thresholds=THRESHOLD_GRID):
    rows = []
    for th in thresholds:
        preds = (probabilities >= th).astype(int)
        metrics = evaluate_sepsis_score_from_arrays(ids, labels, probabilities, preds)
        rows.append({
            "threshold": float(th),
            **metrics
        })
    return pd.DataFrame(rows)


In [54]:
val_prob_eval = final_model.predict(X_val_eval, verbose=0).ravel()

df_utility_val = sweep_thresholds_for_utility(
    id_val_eval,
    y_val_eval,
    val_prob_eval,
    thresholds=THRESHOLD_GRID
).sort_values("utility", ascending=False)

print("Top thresholds by utility on VAL:")
display(df_utility_val.head(10))

best_threshold_utility = float(df_utility_val.iloc[0]["threshold"])
print("Best threshold by utility =", best_threshold_utility)


Top thresholds by utility on VAL:


,threshold,auroc,auprc,accuracy,f_measure,utility
57,0.62,0.811788,0.094094,0.816491,0.117416,0.377805
58,0.63,0.811788,0.094094,0.820002,0.118484,0.377697
56,0.61,0.811788,0.094094,0.812738,0.116253,0.377295
55,0.60,0.811788,0.094094,0.809090,0.115189,0.376834
54,0.59,0.811788,0.094094,0.805296,0.113875,0.375692
59,0.64,0.811788,0.094094,0.823682,0.119184,0.374872
53,0.58,0.811788,0.094094,0.801763,0.112699,0.374479
51,0.56,0.811788,0.094094,0.794735,0.110879,0.373732
67,0.72,0.811788,0.094094,0.854227,0.131088,0.373686
52,0.57,0.811788,0.094094,0.798183,0.111660,0.373262


Best threshold by utility = 0.6200000000000001


In [55]:
test_prob_eval = final_model.predict(X_test_eval, verbose=0).ravel()
test_pred_eval = (test_prob_eval >= best_threshold_utility).astype(int)

utility_metrics_test = evaluate_sepsis_score_from_arrays(
    id_test_eval,
    y_test_eval,
    test_prob_eval,
    test_pred_eval
)

print("=== TEST UTILITY-STYLE RESULT ===")
for k, v in utility_metrics_test.items():
    print(f"{k}: {v:.6f}")


=== TEST UTILITY-STYLE RESULT ===
auroc: 0.832417
auprc: 0.095093
accuracy: 0.818794
f_measure: 0.116726
utility: 0.395758


In [56]:
def export_physionet_psv(ids, labels, probabilities, predictions, out_labels_dir, out_preds_dir):
    os.makedirs(out_labels_dir, exist_ok=True)
    os.makedirs(out_preds_dir, exist_ok=True)

    df = pd.DataFrame({
        "id": ids,
        "SepsisLabel": labels,
        "PredictedProbability": probabilities,
        "PredictedLabel": predictions,
    })

    for pid, g in df.groupby("id", sort=False):
        fname = f"p{int(pid):06d}.psv"

        g[["SepsisLabel"]].to_csv(
            os.path.join(out_labels_dir, fname), sep="|", index=False
        )
        g[["PredictedProbability", "PredictedLabel"]].to_csv(
            os.path.join(out_preds_dir, fname), sep="|", index=False
        )

    print("Exported to:")
    print("labels_dir =", out_labels_dir)
    print("preds_dir  =", out_preds_dir)

export_physionet_psv(
    id_test_eval,
    y_test_eval,
    test_prob_eval,
    test_pred_eval,
    os.path.join(SAVE_DIR, "utility_test_labels"),
    os.path.join(SAVE_DIR, "utility_test_predictions"),
)


Exported to:
labels_dir = /kaggle/working/utility_test_labels
preds_dir  = /kaggle/working/utility_test_predictions
